# 06 · 데이터 전처리

원본 보존 → 품질 점검 → 변환 → 처리 전후 검증 순서로 결측, 중복, 이상치를 다룹니다.

> 위에서 아래로 실행하세요. 예제 데이터는 노트북 안에서 만듭니다. 코드 셀 아래의 출력으로 결과를 확인하고, 실제 데이터에서는 열 이름·단위·기간을 먼저 확인하세요.

## 결측 찾기

**코드 → 코드 개념**: `isna`는 실제 결측을 찾는다. 빈 문자열·특수 숫자는 자동으로 결측이 아니다.

**코드 사용법**: 열별 개수·비율과 위장된 결측을 확인한다.

In [ ]:
import pandas as pd
import numpy as np
raw = pd.DataFrame({"machine": ["A", "A", "A", "B", "B"],
                    "time": [1, 2, 2, 1, 2],
                    "vibration": [2.1, "", "", -999, 15.0]})
print(raw.isna().sum())
raw["vibration"] = pd.to_numeric(raw["vibration"].replace({"": np.nan, -999: np.nan}), errors="coerce")
print(raw.isna().sum(), raw.isna().mean())

**같은 결과를 얻는 방법과 선택 이유**

- `replace`는 알려진 오류 코드를 결측으로 바꾼다. `to_numeric(errors='coerce')`는 변환할 수 없는 모든 값을 결측으로 바꾸므로 무엇이 바뀌었는지 점검한다.
- CSV 읽기 단계에서 `pd.read_csv(..., na_values=['', -999])`를 쓰면 같은 의도를 입력 시점에 반영할 수 있다.

## 행과 열 결측 점검

**코드 → 코드 개념**: `isna().sum(axis=1)`은 행마다, `isna().sum()`은 열마다 결측을 센다.

**코드 사용법**: 삭제 후보 행과 필수 열 결측을 찾는다.

In [ ]:
print(raw.isna().sum(axis=1))
print(raw.loc[raw["vibration"].isna()])
print(raw.dropna(subset=["vibration"]))
print(raw.dropna(thresh=3))

**같은 결과를 얻는 방법과 선택 이유**

- `dropna(subset=...)`는 필수 열이 없는 행만 제거한다. `thresh=3`은 비결측 값이 3개 이상인 행을 남긴다. 전체 `dropna()`는 필요 없는 열의 결측 때문에 행을 과하게 없앨 수 있다.
- 삭제 전후 건수와 설비별 분포를 비교한다.

## 중복 정의와 제거

**코드 → 코드 개념**: `duplicated`는 완전히 같은 행 또는 지정 키의 중복을 표시한다.

**코드 사용법**: 설비·시간 조합의 중복을 확인하고 한 행만 남긴다.

In [ ]:
print(raw.duplicated().sum())
print(raw.duplicated(["machine", "time"], keep=False))
dedup = raw.drop_duplicates(["machine", "time"], keep="first").copy()
print(len(raw), len(dedup))

**같은 결과를 얻는 방법과 선택 이유**

- 전체 행 중복은 재수집된 복사본일 수 있고, 키 중복은 같은 시각의 재측정일 수 있다. 먼저 업무상 키를 정한다.
- `keep='first'`는 현재 순서 기준이다. 수집 시각이나 품질 플래그가 있다면 먼저 정렬한다.

## 결측 대체의 여러 방법

**코드 → 코드 개념**: 평균·중앙값·그룹별 중앙값·앞 값 채우기는 서로 다른 가정이다.

**코드 사용법**: 전역 대체와 설비별 대체를 비교한다.

In [ ]:
clean = dedup.copy()
clean["global_median"] = clean["vibration"].fillna(clean["vibration"].median())
clean["group_median"] = clean.groupby("machine")["vibration"].transform(lambda s: s.fillna(s.median()))
print(clean)
print(clean["group_median"].isna().sum())

**같은 결과를 얻는 방법과 선택 이유**

- 평균은 대칭 분포에, 중앙값은 극단값이 있을 때 상대적으로 견고하다. 설비별 기준이 다르면 그룹별 대체가 자연스럽다.
- 그룹 전체가 결측이면 그룹 중앙값도 결측이다. 임의의 0으로 채우지 말고 대체 규칙과 플래그를 정한다.

## IQR 이상치 후보

**코드 → 코드 개념**: Q1과 Q3 사이 폭인 IQR을 이용해 검토할 범위를 만든다.

**코드 사용법**: 상한·하한을 계산하고 후보를 표시한다.

In [ ]:
x = pd.Series([2.1, 2.4, 2.8, 3.0, 3.2, 3.5, 15.0])
q1, q3 = x.quantile([0.25, 0.75])
lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
mask = x.lt(lower) | x.gt(upper)
print(lower, upper, x[mask].tolist())
print(x.clip(lower=lower, upper=upper).tolist())

**같은 결과를 얻는 방법과 선택 이유**

- `mask`는 원본을 보존하며 후보를 표시한다. `clip`은 값을 경계로 바꾸므로 실제 고장 급등을 숨길 수 있다.
- IQR은 분포 기준이고 설비 안전 한계는 물리 기준이다. 원인 확인 전 자동 삭제하지 않는다.

## 원본 학습 자료

[`1. lecture/02_Pandas/statistics/NaN`](../1.%20lecture/02_Pandas/statistics/NaN), [`4. Summary/07_Pandas`](../4.%20Summary/07_Pandas)